In [92]:
import os
from pathlib import Path
import pandas as pd

# defining paths for notebook
current_dir = Path(globals()["_dh"][0])

raw_data_path = os.path.join(current_dir, r"..\..\raw")
processed_data_path = os.path.join(current_dir, "medals_per_country.csv")
parquet = os.path.join(current_dir, "medals_per_country.parquet")

In [ ]:
df_all_path = os.path.join(raw_data_path, "olympics_history.csv")
df_all = pd.read_csv(df_all_path)

In [ ]:
df_paris_path = os.path.join(raw_data_path, "olympics_paris2024.csv")
df_paris = pd.read_csv(df_paris_path)

# Processing Dataframes

## Processing olympics history

In [52]:
df_all = df_all.filter(["country_noc", "country", "total"], axis=1)
df_all = df_all.groupby(["country_noc", "country"], as_index=False).sum()
df_all.head(10)

,country_noc,country,total
0,AFG,Afghanistan,2
1,AHO,Netherlands Antilles,1
2,ALG,Algeria,17
3,ANZ,Australasia,12
4,ARG,Argentina,77
5,ARM,Armenia,18
6,AUS,Australia,563
7,AUT,Austria,361
8,AZE,Azerbaijan,49
9,BAH,The Bahamas,16


## Processing Paris 2024

In [86]:
df2 = df_paris.filter(["country_code", "country", "Total"], axis=1)
df2 = df2.groupby(["country_code", "country"], as_index=False).sum()

# renaming
df2 = df2.rename(columns={"country_code": "country_noc", "Total": "total"})
df2["country"] = df2["country"].replace(
    {
        "China": "People's Republic of China",
        "IR Iran": "Islamic Republic of Iran",
        "Korea": "Republic of Korea",
        "DPR Korea": "Democratic People's Republic of Korea",
    },
)
df2.head(20)

,country_noc,country,total
0,AIN,AIN,5
1,ALB,Albania,2
2,ALG,Algeria,3
3,ARG,Argentina,3
4,ARM,Armenia,4
5,AUS,Australia,53
6,AUT,Austria,5
7,AZE,Azerbaijan,7
8,BEL,Belgium,10
9,BOT,Botswana,2


In [87]:
df_paris = df2

# Merging both Dataframes

In [88]:
df_final = pd.concat([df_all, df_paris])
df_final = df_final.groupby(["country_noc", "country"]).sum()
df_final.head(10)

,,total
country_noc,country,
AFG,Afghanistan,2
AHO,Netherlands Antilles,1
AIN,AIN,5
ALB,Albania,2
ALG,Algeria,20
ANZ,Australasia,12
ARG,Argentina,80
ARM,Armenia,22
AUS,Australia,616


## Saving files

In [93]:
df_final.to_csv(processed_data_path)
df_final.to_parquet(parquet, index=False, engine="fastparquet")